In [1]:
import pandas as pd
import re
import numpy as np
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

data = pd.read_csv("twitter_dataset.csv")

data.head()

,Tweet_ID,Username,Text,Retweets,Likes,Timestamp
0,1,julie81,Party least receive say or single. Prevent pre...,2,25,2023-01-30 11:00:51
1,2,richardhester,Hotel still Congress may member staff. Media d...,35,29,2023-01-02 22:45:58
2,3,williamsjoseph,Nice be her debate industry that year. Film wh...,51,25,2023-01-18 11:25:19
3,4,danielsmary,Laugh explain situation career occur serious. ...,37,18,2023-04-10 22:06:29
4,5,carlwarren,Involve sense former often approach government...,27,80,2023-01-24 07:12:21


In [2]:
def clean_tweet(tweet):
    tweet = str(tweet).lower()
    tweet = re.sub(r"http\S+|www\S+", "", tweet)
    tweet = re.sub(r"@\w+", "", tweet)
    tweet = re.sub(r"#", "", tweet)
    tweet = re.sub(r"[^a-z\s]", " ", tweet)

    return tweet.split()

sentences = data["Text"].apply(clean_tweet).tolist()

print(sentences[:5])

[['party', 'least', 'receive', 'say', 'or', 'single', 'prevent', 'prevent', 'husband', 'affect', 'may', 'himself', 'cup', 'style', 'evening', 'protect', 'effect', 'another', 'themselves', 'stage', 'perform', 'possible', 'try', 'tax', 'share', 'style', 'television', 'with', 'successful', 'much', 'sell', 'development', 'economy', 'effect'], ['hotel', 'still', 'congress', 'may', 'member', 'staff', 'media', 'draw', 'buy', 'fly', 'identify', 'on', 'another', 'turn', 'minute', 'would', 'local', 'subject', 'way', 'believe', 'which', 'question', 'some', 'message', 'own', 'all', 'imagine', 'join', 'agency', 'indicate'], ['nice', 'be', 'her', 'debate', 'industry', 'that', 'year', 'film', 'where', 'generation', 'push', 'discover', 'partner', 'level', 'nearly', 'money', 'store', 'style', 'may', 'enjoy', 'kid', 'discuss', 'blue', 'save', 'model', 'another', 'about', 'along', 'everybody', 'especially', 'dinner', 'character', 'yard'], ['laugh', 'explain', 'situation', 'career', 'occur', 'serious', 'f

In [3]:
cbow = Word2Vec(
    sentences=sentences,
    vector_size=50,
    window=5,
    min_count=2,
    sg=0,
    epochs=5
)

In [4]:
def get_document_vector(words, model):
    vectors = [model.wv[word] for word in words if word in model.wv]

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

document_vectors = np.array([
    get_document_vector(words, cbow)
    for words in sentences
])

print("Document vectors shape:", document_vectors.shape)

Document vectors shape: (10000, 50)


In [5]:
user_input = input("Enter a topic or search query: ")

user_words = clean_tweet(user_input)

user_vector = get_document_vector(user_words, cbow)

print("User input vector:")
print(user_vector)

Enter a topic or search query:  history


User input vector:
[-0.0965921  -0.11590612 -0.01243648  0.03892009 -0.31115893 -0.02783101
  0.11843932  0.15817215 -0.34942514  0.00585322 -0.05546648 -0.10030588
 -0.09738095 -0.15246853 -0.31086102 -0.11326434  0.24782743 -0.19318505
 -0.29593387 -0.14312638 -0.13317855  0.02415765  0.32914385 -0.11838644
 -0.04118036  0.04193195 -0.04533605 -0.4834019  -0.3831679   0.20112363
  0.16244926 -0.06000542 -0.11279234  0.27451375 -0.17239991  0.3709938
 -0.38821787  0.00781948 -0.01044494 -0.42302692  0.36713582  0.01049747
 -0.21552981 -0.00388222  0.28880876  0.11434729 -0.13615231 -0.17774689
  0.12776504  0.03422142]


In [6]:
similarities = cosine_similarity(
    user_vector.reshape(1, -1),
    document_vectors
)[0]

print(similarities[:5])

[0.99625117 0.99582636 0.99607265 0.9962685  0.9960528 ]


In [7]:
top_indices = similarities.argsort()[::-1][:5]

print("Recommended Tweets:\n")

for index in top_indices:
    print(data["Text"].iloc[index])
    print("Similarity:", similarities[index])
    print()

Recommended Tweets:

Difficult head then campaign. Rule student drop. Question growth along issue attention. Worry human ground discussion let lay.
Realize anyone treat act shake. Great threat spend walk leave history.
Similarity: 0.9967404

Where true church dog record particular. Campaign anyone simple success. Near term design some situation.
Similarity: 0.9967233

History student baby shoulder understand. Surface child bit grow.
Government bar reduce. Democratic officer finish ability suddenly. Hand our poor forget wait chance history per.
Similarity: 0.996705

Or modern week ground eye have religious. Star him community.
All interview cold heavy open laugh remain. Brother late teach history marriage. Player rise Mr choose.
Similarity: 0.996698

Boy modern subject state writer free region bag.
Ability ask day factor we certainly compare use. Region far kind.
Home finish Mr need heavy government. Up necessary system compare.
Similarity: 0.9966725

